In [1]:
import numpy as np
import pandas as pd

from subprocess import check_output
print(check_output('dir .\\input', shell=True).decode('cp949'))

 C 드라이브의 볼륨에는 이름이 없습니다.
 볼륨 일련 번호: 2602-0473

 c:\Users\USER\Desktop\AI_study\kaggle_transcription\Binary classification - Image classification\1st level. Statoil C-CORE Iceberg Classifier Challenge\input 디렉터리

2026-04-07  오후 01:29    <DIR>          .
2026-04-10  오후 08:44    <DIR>          ..
2026-04-07  오후 01:29    <DIR>          data
2017-10-24  오전 02:27           117,951 sample_submission.csv
2026-03-07  오후 08:56            38,566 sample_submission.csv.7z
2017-10-24  오전 02:27     1,521,771,850 test.json
2026-03-07  오후 08:56       257,127,394 test.json.7z
2017-10-24  오전 02:23       196,313,674 train.json
2026-03-07  오후 08:56        44,932,785 train.json.7z
               6개 파일       2,020,302,220 바이트
               3개 디렉터리  1,393,823,481,856 바이트 남음



TL;DR
**Runs on GPU** There is some compatibility issue with CPUs

1. Hyperparameters in Deep learning are many, tuning them will take weeks or months. Generally researchers do this tuning and publish paper when they find a nice set of architecture which performs better than other.
2. Since the model is pre-trained, it converges very fast and you but still you need GPU to use this. Due to some library issues, it doesn't work on CPU.
3. For our purpose, we can use those architectures, which are made available by those researchers to us.
4. Using those pretrained nets, layers of which already 'knows' how to extract features, we can don't have to tune the hyperparameters. Since they are already trained of some dataset(say imagenet), their pre-trained weights provide a good initialization of weights and because of this, our Convnet converges very fast which otherwise can take days on these deep architectures. That's the idea behind **Transfer Learning**. Examples of which are VGG16, InceptionNet, googlenet, Resnet etc.
5. In this kernel we will use pretrained VGG-16 network which performs very well on small size images.
6. **VGG Architecture has proved to worked well on small sized images(CIFAR-10)** I expected it to work well for this dataset as well.
   1. The code also includes the data augmentation steps, thus considerably improving the performance.
   2. **GPU is needed**

Here is the link of the research paper if you are interested. https://arxiv.org/pdf/1409.1556.pdf

Also here is the doc for keras library: https://keras.io/applications/#vgg16

In [2]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import log_loss
from sklearn.model_selection import StratifiedKFold, StratifiedShuffleSplit
from os.path import join as opj
from matplotlib import pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
import pylab
plt.rcParams['figure.figsize'] = 10, 10
%matplotlib inline

In [3]:
train = pd.read_json('./input/train.json')
target_train = train['is_iceberg']
test = pd.read_json('./input/test.json')

Keras provide the implementation of pretrained VGG, it in it's library so we don't have to build the net by ourselves. Here we are removing the last layer of VGG and putting our sigmoid layer for binary predictions.

The following code will NOT WORK, since on kaggle notebook, the weights of model cannot be downloaded, however, you can copy paste the code in your own notebook to make it work.

In [13]:
target_train = train['is_iceberg']
test['inc_angle'] = pd.to_numeric(test['inc_angle'], errors='coerce')
train['inc_angle'] = pd.to_numeric(train['inc_angle'], errors='coerce')
train['inc_angle'] = train['inc_angle'].fillna(method='pad')
X_angle = train['inc_angle']
test['inc_angle'] = pd.to_numeric(test['inc_angle'], errors='coerce')
X_test_angle = test['inc_angle']

X_band_1 = np.array([np.array(band).astype(np.float32).reshape(75, 75) for band in train['band_1']])
X_band_2 = np.array([np.array(band).astype(np.float32).reshape(75, 75) for band in train['band_2']])
X_band_3 = (X_band_1 + X_band_2) / 2

X_train = np.concatenate([X_band_1[:, :, :, np.newaxis], X_band_2[:, :, :, np.newaxis], X_band_3[:, :, :, np.newaxis]], axis=-1)

X_band_test_1 = np.array([np.array(band).astype(np.float32).reshape(75, 75) for band in test['band_1']])
X_band_test_2 = np.array([np.array(band).astype(np.float32).reshape(75, 75) for band in test['band_2']])
X_band_test_3 = (X_band_test_1 + X_band_test_2) / 2

X_test = np.concatenate([X_band_test_1[:, :, :, np.newaxis], X_band_test_2[:, :, :, np.newaxis], X_band_test_3[:, :, :, np.newaxis]], axis=-1)

from matplotlib import pyplot
from tf_keras.optimizers import RMSprop
from tf_keras.preprocessing.image import ImageDataGenerator
from tf_keras.models import Sequential
from tf_keras.layers import Conv2D, MaxPooling2D, Dense, Dropout, Input, Flatten, Activation
from tf_keras.layers import GlobalMaxPooling2D
from tf_keras.layers import BatchNormalization
from tf_keras.layers import Concatenate
from tf_keras.models import Model
from tf_keras import initializers
from tf_keras.optimizers import Adam
from tf_keras.optimizers import RMSprop
from tf_keras.layers import LeakyReLU, PReLU
from tf_keras.optimizers.legacy import SGD
from tf_keras.callbacks import ModelCheckpoint, Callback, EarlyStopping

from tf_keras.datasets import cifar10
from tf_keras.applications.inception_v3 import InceptionV3
from tf_keras.applications.vgg16 import VGG16
from tf_keras.applications.xception import Xception
from tf_keras.applications.mobilenet import MobileNet
from tf_keras.applications.vgg19 import VGG19
from tf_keras.layers import concatenate, Dense, LSTM, Input, Concatenate
from tf_keras.preprocessing import image
from tf_keras.applications.vgg16 import preprocess_input

batch_size = 64

gen = ImageDataGenerator(horizontal_flip=True,
                         vertical_flip=True,
                         width_shift_range=0.,
                         height_shift_range=0.,
                         channel_shift_range=0,
                         zoom_range=0.2,
                         rotation_range=10)

def gen_flow_for_two_inputs(X1, X2, y):
    genX1 = gen.flow(X1, y, batch_size=batch_size, seed=55)
    genX2 = gen.flow(X1, X2, batch_size=batch_size, seed=55)
    while True:
        X1i = genX1.next()
        X2i = genX2.next()

        yield (X1i[0], X2i[1]), X1i[1]

def get_callbacks(filepath, patience=2):
    es = EarlyStopping('val_loss', patience=10, mode='min')
    msave = ModelCheckpoint(filepath, save_best_only=True)
    return [es, msave]

def getVggAngleModel():
    input_2 = Input(shape=[1], name='angle')
    angle_layer = Dense(1, )(input_2)
    base_model = VGG16(weights='imagenet', include_top=False, input_shape=X_train.shape[1:], classes=1)
    x = base_model.get_layer('block5_pool').output

    x = GlobalMaxPooling2D()(x)
    merge_one = concatenate([x, angle_layer])
    merge_one = Dense(512, activation='relu', name='fc2')(merge_one)
    merge_one = Dropout(0.3)(merge_one)
    merge_one = Dense(512, activation='relu', name='fc3')(merge_one)
    merge_one = Dropout(0.3)(merge_one)

    predictions = Dense(1, activation='sigmoid')(merge_one)

    # model = Model(input=[base_model.input, input_2], output=predictions)
    model = Model(inputs=[base_model.input, input_2], outputs=predictions)
    
    sgd = SGD(learning_rate=1e-3, decay=1e-6, momentum=0.9, nesterov=True)
    model.compile(loss='binary_crossentropy',
                  optimizer=sgd,
                  metrics=['accuracy'])
    return model

def myAngleCV(X_train, X_angle, X_test):
    K=3
    folds = list(StratifiedKFold(n_splits=K, shuffle=True, random_state=16).split(X_train, target_train))
    y_test_pred_log = 0
    y_train_pred_log = 0
    y_valid_pred_log = 0.0 * target_train
    for j, (train_idx, test_idx) in enumerate(folds):
        print('\n======================FOLD=', j)
        X_train_cv = X_train[train_idx]
        y_train_cv = target_train[train_idx]
        X_holdout = X_train[test_idx]
        Y_holdout = target_train[test_idx]

        X_angle_cv = X_angle[train_idx]
        X_angle_hold = X_angle[test_idx]

        file_path = '%s_aug_model_weights.hdf5'%j
        callbacks = get_callbacks(filepath=file_path, patience=5)
        gen_flow = gen_flow_for_two_inputs(X_train_cv, X_angle_cv, y_train_cv)
        galaxyModel = getVggAngleModel()
        galaxyModel.fit(
            gen_flow,
            steps_per_epoch=24,
            epochs=100,
            shuffle=True,
            verbose=1,
            validation_data=([X_holdout, X_angle_hold], Y_holdout),
            callbacks=callbacks
        )

        galaxyModel.load_weights(filepath=file_path)
        
        score = galaxyModel.evaluate([X_train_cv, X_angle_cv], y_train_cv, verbose=0)
        print('Train loss:', score[0])
        print('Train accuracy:', score[1])

        score = galaxyModel.evaluate([X_holdout, X_angle_hold], Y_holdout, verbose=0)
        print('Test loss:', score[0])
        print('Test accuracy:', score[1])

        pred_valid = galaxyModel.predict([X_holdout, X_angle_hold])
        y_valid_pred_log[test_idx] = pred_valid.reshape(pred_valid.shape[0])

        temp_test = galaxyModel.predict([X_test, X_test_angle])
        y_test_pred_log += temp_test.reshape(temp_test.shape[0])

        temp_train = galaxyModel.predict([X_train, X_angle])
        y_train_pred_log += temp_train.reshape(temp_train.shape[0])

    y_test_pred_log = y_test_pred_log / K
    y_train_pred_log = y_train_pred_log / K

    print('\n Train Log Loss Validation= ', log_loss(target_train, y_train_pred_log))
    print('  Test Log Loss Validation= ', log_loss(target_train, y_valid_pred_log))
    return y_test_pred_log

C:\Users\USER\AppData\Local\Temp\ipykernel_7652\1259300295.py:4: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  train['inc_angle'] = train['inc_angle'].fillna(method='pad')


In [14]:
preds = myAngleCV(X_train, X_angle, X_test)


======================FOLD= 0
Epoch 1/100



 7/24 [=======>......................] - ETA: 20s - loss: 0.8909 - accuracy: 0.5156

KeyboardInterrupt: 

In [ ]:
submission = pd.DataFrame()
submission['id'] = test['id']
submission['is_iceberg'] = preds
submission.to_csv('sub.csv', index=False)